# Experiment 7: Morphological Analysis of English Sentences
**Mini-project:** Resume Analyzer Using Natural Language Processing (Group 28)

**Objective:** Analyse the morphology of resume sentences: lemma, part of speech, inflectional features, stem and prefix/suffix segmentation, and study inflection and derivation across the corpus.

Open in Colab: [Exp7_Morphological_Analysis.ipynb](https://colab.research.google.com/github/GnsShubham06/resume-analyzer-nlp/blob/main/notebooks/experiments/Exp7_Morphological_Analysis.ipynb) &nbsp;|&nbsp; Run with **Runtime → Run all**.
Data: 100 resumes (20 per domain); the screenshots in the report use five fictional (synthetic) resumes.

In [ ]:
# ---- Setup: runs in Google Colab and locally --------------------------------
# In Colab this cell downloads the project repository (code + anonymised
# dataset) from GitHub and installs the packages Colab does not ship with.
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
REPO = "https://github.com/GnsShubham06/resume-analyzer-nlp"
if "google.colab" in sys.modules:
    if not os.path.isdir("/content/resume-analyzer-nlp"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/content/resume-analyzer-nlp"], check=True)
    os.chdir("/content/resume-analyzer-nlp")
    extra = []
    if extra:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra], check=True)
else:  # local run from notebooks/experiments/
    for _ in range(3):
        if os.path.isfile("requirements.txt"):
            break
        os.chdir("..")
sys.path.insert(0, os.getcwd())

import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)

import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:  # model not installed yet
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

%config InlineBackend.figure_format = "retina"
import pandas as pd
pd.set_option("display.max_colwidth", 120)
df = pd.read_csv("data/resume_dataset.csv")
df["extracted_text"] = df["extracted_text"].fillna("")
print("Project folder:", os.path.basename(os.getcwd()), "| resumes:", len(df))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10})
ORDER = ["Engineering", "HR", "Finance", "Healthcare", "Agriculture"]
SAMPLE_IDS = {"Engineering": "ENG_06", "HR": "HR_06", "Finance": "FIN_07",
              "Healthcare": "HLT_04", "Agriculture": "AGR_02"}     # five fictional resumes
splits = pd.read_csv("data/processed/splits.csv")[["resume_id", "split"]]
df = df.merge(splits, on="resume_id")

import re
from collections import Counter, defaultdict
from nltk.corpus import wordnet as wn
from nltk.stem import PorterStemmer
ps = PorterStemmer()
PREFIXES = ["multi", "inter", "trans", "under", "over", "super", "micro", "non", "pre", "re", "un",
            "dis", "mis", "co", "sub", "in", "im"]
DERIV = ["ization", "ational", "ation", "ition", "ment", "ness", "ance", "ence", "ship", "ity", "ism", "ist",
         "ive", "able", "ible", "ful", "less", "ous", "ical", "al", "ize", "ise", "ify", "ee", "er", "or", "ly", "ing"]
INFL = ["ing", "ed", "es", "s", "d"]
NEG = {"un", "mis", "dis", "non"}
def segment(word, lemma):
    """prefix + root + derivational suffix (found on the lemma) + inflectional suffix (word vs lemma)."""
    w, lem = word.lower(), lemma.lower()
    infl = next((s for s in INFL if w != lem and w.endswith(s)), "")
    # a prefix counts only if the rest is itself an English word (re+organise, mis+file, but not re+cord)
    pre = next((p for p in PREFIXES if lem.startswith(p) and wn.synsets(lem[len(p):])
                and len(lem) - len(p) >= (4 if p in NEG else 5)), "")
    rest = lem[len(pre):]
    deriv = next((s for s in DERIV if rest.endswith(s) and len(rest) > len(s) + 3), "")
    root = rest[:-len(deriv)] if deriv else rest
    parts = ([pre + "-"] if pre else []) + [root] + (["-" + deriv] if deriv else []) + (["-" + infl] if infl else [])
    return pre, deriv, infl, " + ".join(parts)
def analyse(sentence):
    rows = []
    for t in nlp(sentence):
        if t.is_punct:
            continue
        pre, deriv, infl, seg = segment(t.text, t.lemma_)
        rows.append((t.text, t.lemma_, t.pos_, str(t.morph) or "-", ps.stem(t.text), seg))
    return pd.DataFrame(rows, columns=["word", "lemma", "POS", "morphological features (spaCy)",
                                       "Porter stem", "segmentation"])

## a. Word-level analysis of a resume sentence

In [ ]:
s = "Organised farmer field schools and demonstrations on zero-budget natural farming."   # AGR_02 (fictional)
print("Sentence:", s)
display(analyse(s))

In [ ]:
s2 = "Reorganised the misfiled employee records and handled unresolved benefit queries."
print("Sentence (prefixes):", s2)
display(analyse(s2)[["word", "lemma", "POS", "morphological features (spaCy)", "segmentation"]])

## b. Inflection and derivation across all 100 resumes

In [ ]:
infl_c, deriv_c, feats = Counter(), Counter(), Counter()
word_info = defaultdict(Counter)          # word -> Counter of (POS, lemma) seen in the corpus
n_words = 0
for doc in nlp.pipe(df.cleaned_text.str.slice(0, 20000), batch_size=8):
    for t in doc:
        if not t.is_alpha:
            continue
        n_words += 1
        _, deriv, infl, _ = segment(t.text, t.lemma_)
        if infl: infl_c["-" + infl] += 1
        if deriv: deriv_c["-" + deriv] += 1
        for f in t.morph: feats[f] += 1
        word_info[t.text.lower()][(t.pos_, t.lemma_.lower())] += 1
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6))
for ax, (title, c) in zip(axes, [("inflectional suffixes", infl_c), ("top derivational suffixes", deriv_c)]):
    items = [(k, v) for k, v in c.most_common(8) if v >= 10][::-1]
    bars = ax.barh([k for k, _ in items], [v for _, v in items], color="#555555")
    ax.bar_label(bars, padding=2, fontsize=7); ax.set_title(title)
plt.tight_layout(); plt.show()
print(f"{n_words:,} alphabetic words analysed")
print("most frequent spaCy features:", feats.most_common(6))

In [ ]:
rows = []
for root, base in [("manag", "manage"), ("employ", "employ"), ("recruit", "recruit"), ("supervis", "supervise"),
                   ("coordinat", "coordinate"), ("analy", "analyse")]:
    def best(w):   # most frequent analysis of a word, ignoring PROPN tags of capitalised headings
        common = [k for k, _ in word_info[w].most_common() if k[0] != "PROPN"]
        return common[0] if common else None
    family = [w for w in word_info if w.startswith(root) and best(w)]
    infl = sorted(w for w in family if best(w)[1] == base)
    deriv = sorted({f"{best(w)[1]} ({best(w)[0]})" for w in family if best(w)[1] != base})
    rows.append((base, ", ".join(infl), ", ".join(deriv[:5])))
display(pd.DataFrame(rows, columns=["base (verb)", "inflected forms – same lemma", "derived words – new lemma (POS)"]))